# Editorial Review Board | Parallelization

In [2]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [3]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [4]:
model = ChatOpenAI(model="gpt-4o")

In [5]:
class ParallelState(TypedDict):
    input: str
    tone_analysis: str
    fact_check: str
    grammar_review: str
    final_report: str

In [6]:
# Parallel workers
def analyze_tone(state: ParallelState) -> dict:
    response = model.invoke(f"Analyze the tone of this text:\n\n{state['input']}")
    return {"tone_analysis": response.content}

def check_facts(state: ParallelState) -> dict:
    response = model.invoke(f"Fact-check this text:\n\n{state['input']}")
    return {"fact_check": response.content}

def review_grammar(state: ParallelState) -> dict:
    response = model.invoke(f"Review grammar and style:\n\n{state['input']}")
    return {"grammar_review": response.content}

# Aggregator
def aggregate_results(state: ParallelState) -> dict:
    response = model.invoke(
        f"Combine these reviews into a single editorial report:\n\n"
        f"Tone: {state['tone_analysis']}\n\n"
        f"Facts: {state['fact_check']}\n\n"
        f"Grammar: {state['grammar_review']}"
    )
    return {"final_report": response.content}

In [7]:
# Build the graph with parallel fan-out
graph = StateGraph(ParallelState)
graph.add_node("tone", analyze_tone)
graph.add_node("facts", check_facts)
graph.add_node("grammar", review_grammar)
graph.add_node("aggregate", aggregate_results)

# Fan out from START to all three workers (parallel fan-out supported in LangGraph v1.x)
graph.add_edge(START, "tone")
graph.add_edge(START, "facts")
graph.add_edge(START, "grammar")

# All workers converge to aggregator (LangGraph waits for all branches before running it)
graph.add_edge("tone", "aggregate")
graph.add_edge("facts", "aggregate")
graph.add_edge("grammar", "aggregate")
graph.add_edge("aggregate", END)

parallel = graph.compile()

In [8]:
# Plot the workflow
plot_mermaid(parallel)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	tone(tone)
	facts(facts)
	grammar(grammar)
	aggregate(aggregate)
	__end__([<p>__end__</p>]):::last
	__start__ --> facts;
	__start__ --> grammar;
	__start__ --> tone;
	facts --> aggregate;
	grammar --> aggregate;
	tone --> aggregate;
	aggregate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
result = parallel.invoke({"input": "AI will replace all jobs by 2030."})
print(result["final_report"])

**Editorial Report: Evaluating the Impact of AI on the Job Market by 2030**

The statement "AI will replace all jobs by 2030" can be interpreted as alarmist and speculative, suggesting a definitive and drastic outcome that may provoke concern or fear. This perspective lacks nuance and fails to consider potential positive impacts, such as job transformation or the emergence of new job categories. It conveys an exaggerated sense of inevitability and urgency regarding AI's influence on the job market.

**Factual Evaluation:**
The suggestion that AI will replace all jobs by 2030 is overly simplistic and not supported by current evidence or expert consensus. AI and automation are indeed expected to substantially affect the job market, leading to significant transformations across numerous industries. However, it is improbable that AI will entirely replace all jobs. Several important aspects challenge this claim:

1. **Job Transformation vs. Replacement**: AI is more inclined to transform ex

In [11]:
stream_invoke(parallel, {"input": "AI will replace all jobs by 2030."})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'input': 'AI will replace all jobs by 2030.',
 'tone_analysis': 'The tone of the text "AI will replace all jobs by 2030" can be interpreted as alarmist or speculative. It suggests a definitive and drastic outcome that may provoke concern or fear. The statement lacks nuance and does not account for potential positive aspects or alternatives, such as job transformation or the creation of new job categories. Overall, it projects a sense of inevitability and urgency regarding the impact of AI on the job market.',
 'fact_check': "The claim that AI will replace all jobs by 2030 is overly simplistic and not supported by current evidence or expert consensus. While AI and automation are expected to significantly impact the job market, leading to the transformation of many industries and job roles, it's unlikely that all jobs will be replaced by AI.\n\nHere are several points to consider:\n\n1. **Job Transformation vs. Replacement**: AI is more likely to transform jobs rather than completely re